# Lecture 10 — A supervisor with calendar and email workers

Run cells from top to bottom. The default path uses local data and an explicit
scripted chat-model test double inside real LangChain/LangGraph agent loops.
No API key, hosted trace, external calendar or external email service is used.
All essential source code is included here. Restarting the kernel clears the
in-memory checkpoints and local records.

Tested with Python 3.14; langchain 1.4.7, langgraph 1.2.16,
langchain-core 1.7.0 and pydantic 2.13.5. Optional live mode was not executed.
In a fresh notebook environment, run this installation command once, then
restart the kernel before running the programs:

```python
%pip install langchain==1.4.7 langgraph==1.2.16 langchain-core==1.7.0 pydantic==2.13.5
```

Only if you intend to use optional live mode, also install:

```python
%pip install langchain-openrouter==0.2.9 openrouter==0.11.46
```

The task is to find a free design-team hour after 14:00 on Tuesday,
27 October 2026, create a local event, and record an email reminder using
the resulting event. The trusted clock is Sunday, 25 October 2026, at
12:00 in Europe/Kyiv. The two contacts use reserved `.test` addresses.

## Lecture map

The lecture sections below describe the learning path. Program numbers identify
the runnable order; they are separate from lecture section numbers.

1. **Decide whether several agents will help**
   - 1.1 One request, several kinds of work.
   - 1.2 Diagnose tool and context problems before adding agents.
2. **Choose how the work moves**
   - 2.1 Parallel work and sequential dependencies.
   - 2.2 Improve a draft in a loop; combine several results.
   - 2.3 Route once or supervise repeatedly.
   - 2.4 Handoffs, networks and hierarchies.
   - 2.5 Combine patterns around the actual task.
3. **Build a supervisor with two specialists**
   - 3.1 The libraries and the three layers.
   - 3.2 Set up a local calendar and email workspace.
   - 3.3 Create workers, wrap them as tools, then add the supervisor.
   - 3.4 Run the workers and read the supervisor's execution record.
4. **Give each agent the information it needs**
   - 4.1 Names and descriptions guide delegation.
   - 4.2 Select inputs and inject trusted application context.
   - 4.3 Return useful evidence and update state explicitly.
5. **Pause before actions and review the exact proposal**
   - 5.1 Put the approval gate at the tool boundary.
   - 5.2 Inspect a pause; approve or edit the action.
   - 5.3 Reject an action and verify what did not happen.
6. **Test the system and decide whether it is worth using**
   - 6.1 Check each layer and the complete job.
   - 6.2 Optional real models and execution records.
   - 6.3 Synthesis, application checks and references.


## Program 1. Imports and a trusted local workspace

Use an explicit reference clock, date, timezone and fictional directory. The two lists record local calendar and simulated-send effects only.

Prerequisites: Python 3.11+; langchain, langgraph and their compatible langchain-core.

In [1]:
import json
import os
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import NotRequired
from zoneinfo import ZoneInfo
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Prevent inherited hosted tracing during this entirely local lesson.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

@dataclass(frozen=True)
class TrustedContext:
    date: str = "2026-10-27"
    timezone: str = "Europe/Kyiv"
    reference_clock: str = "2026-10-25T12:00:00+02:00"
    attendees: tuple = ("maya@example.test", "leo@example.test")

CONTEXT = TrustedContext()
# Half-open busy intervals: an event ending at 15:00 permits a 15:00 start.
BUSY = {CONTEXT.attendees[0]: [(9, 10), (11, 12), (13, 14)],
        CONTEXT.attendees[1]: [(10, 12), (14, 15)]}

@dataclass
class Workspace:
    events: list = field(default_factory=list)
    outbox: list = field(default_factory=list)
    packets: list = field(default_factory=list)

class ParentState(AgentState):
    calendar_result: NotRequired[dict]
    email_result: NotRequired[dict]

print("Local fixture:", CONTEXT.date, CONTEXT.timezone)


Local fixture: 2026-10-27 Europe/Kyiv


## Program 2. Compute availability and validate local effects

Ordinary Python computes free hours. Tools validate the exact arguments that will run, including human edits. A sent status means a local outbox record.

Prerequisites: Run program 1.

In [2]:
def slots(attendees, date, not_before_hour=14):
    if set(attendees) != set(CONTEXT.attendees) or date != CONTEXT.date:
        raise ValueError("Use the trusted design-team directory and date.")
    zone = ZoneInfo(CONTEXT.timezone)
    candidates = []
    for hour in range(max(9, not_before_hour), 17):
        free = all(not (hour < end and hour + 1 > start)
                   for person in attendees for start, end in BUSY[person])
        if free:
            start = datetime.fromisoformat(f"{date}T{hour:02}:00").replace(
                tzinfo=zone)
            candidates.append({
                "start": start.isoformat(),
                "end": (start + timedelta(hours=1)).isoformat()
            })
    return candidates

def local_tools(workspace):
    def evidence(result):
        return json.dumps(result), result

    @tool(response_format="content_and_artifact")
    def get_available_time_slots(attendees: list[str], date: str,
                                 not_before_hour: int = 14) -> tuple:
        """Find mutually free one-hour slots on the trusted local calendar."""
        return evidence({"slots": slots(attendees, date, not_before_hour)})

    @tool(response_format="content_and_artifact")
    def create_calendar_event(title: str, start: str, end: str,
                              attendees: list[str]) -> tuple:
        """Create one local event after exact time and recipient validation."""
        if {"start": start, "end": end} not in slots(attendees, CONTEXT.date):
            raise ValueError("Choose a complete available one-hour slot.")
        if any(e["start"] == start for e in workspace.events):
            raise ValueError("That local slot has already been booked.")
        event = dict(status="scheduled", id=f"event-{len(workspace.events)+1}",
                     title=title, start=start, end=end, attendees=attendees,
                     timezone=CONTEXT.timezone)
        workspace.events.append(event)
        return evidence(event)

    @tool(response_format="content_and_artifact")
    def send_email(to: list[str], subject: str, body: str,
                   event_id: str) -> tuple:
        """Record a simulated email locally for an existing calendar event."""
        event = next(e for e in workspace.events if e["id"] == event_id)
        if set(to) != set(event["attendees"]):
            raise ValueError("Recipients must match the scheduled event.")
        if event["start"] not in body or event["end"] not in body:
            raise ValueError("The message must use the actual event times.")
        message = dict(status="sent_locally", to=to, subject=subject,
                       body=body, event_id=event_id)
        workspace.outbox.append(message)
        return evidence(message)

    return get_available_time_slots, create_calendar_event, send_email

print("Computed free hours:", [s["start"] for s in slots(
    list(CONTEXT.attendees), CONTEXT.date)])


Computed free hours: ['2026-10-27T15:00:00+02:00', '2026-10-27T16:00:00+02:00']


## Program 3. A scripted model that exercises real tool-call loops

This test double chooses a fixed sequence from its role and message history. It emits real AIMessage tool calls; create_agent executes them. It does not test language understanding, planning quality or model intelligence.

Prerequisites: Run programs 1–2.

In [3]:
class ScriptedModel(BaseChatModel):
    role: str

    @property
    def _llm_type(self):
        return "lecture10-scripted-test-double"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        user = next(m for m in reversed(messages)
                    if isinstance(m, HumanMessage))
        history = messages[messages.index(user) + 1:]
        results = [m for m in history if isinstance(m, ToolMessage)]
        action, args = None, {}
        if self.role == "supervisor":
            if not results:
                action = "schedule_event"
            elif len(results) == 1 and "email" in user.content.lower():
                event = results[-1].artifact
                if event.get("status") == "scheduled":
                    action = "manage_email"
            final = {m.name: m.artifact for m in results}
        else:
            packet = json.loads(user.content)
            if self.role == "calendar" and not results:
                action = "get_available_time_slots"
                args = {k: packet[k] for k in ("attendees", "date")}
            elif self.role == "calendar" and len(results) == 1:
                free = results[-1].artifact["slots"]
                if free:
                    action = "create_calendar_event"
                    args = dict(title="Design review", **free[0],
                                attendees=packet["attendees"])
            elif self.role == "email" and not results:
                event = packet["event"]
                action = "send_email"
                args = dict(
                    to=event["attendees"], subject="Design reminder",
                    body=f"Design review: {event['start']} to {event['end']}",
                    event_id=event["id"])
            final = (results[-1].artifact if results else {})
        if action:
            message = AIMessage(content="", tool_calls=[{
                "name": action, "args": args,
                "id": f"{self.role}-{len(results)}", "type": "tool_call"}])
        else:
            message = AIMessage(content=json.dumps(final))
        return ChatResult(generations=[ChatGeneration(message=message)])


## Program 4. Workers, selected inputs and the supervisor

The supervisor sees two high-level tools. Each wrapper sends a selected packet to a worker and updates parent state using Command plus a ToolMessage with the current call ID. Workers inherit parent checkpointing when called inside the wrapper. The email boundary also requires a real scheduled result.

Prerequisites: Run programs 1–3. ToolRuntime injects application state and trusted context.

In [4]:
def worker_result(result, tool_name):
    message = next((m for m in reversed(result["messages"])
                    if isinstance(m, ToolMessage) and m.name == tool_name),
                   None)
    if message is None:
        return {
            "status": "unavailable", "reason": "No effect tool was executed."
        }
    if message.status == "error":
        rejected = str(
            message.content).startswith("User rejected the tool call")
        return {"status": "rejected" if rejected else "failed",
                "reason": message.content}
    return message.artifact

def build_system(model_factory=None, gated=True):
    factory = model_factory or (lambda role: ScriptedModel(role=role))
    workspace = Workspace()
    availability, create_event, email_tool = local_tools(workspace)

    def gate(name):
        policy = {name: {"allowed_decisions": ["approve", "edit", "reject"]}}
        return [HumanInTheLoopMiddleware(interrupt_on=policy)] if gated else []

    calendar = create_agent(
        factory("calendar"), [availability, create_event],
        system_prompt=("Find a free hour, then create the local "
                       "calendar event."),
        middleware=gate("create_calendar_event"), name="calendar")
    email = create_agent(
        factory("email"), [email_tool],
        system_prompt=("Send one local reminder for the supplied event. "
                       "Preserve its exact ISO start and end strings in "
                       "the email body."),
        middleware=gate("send_email"), name="email")

    def run_worker(worker, packet, result_tool, field, runtime):
        workspace.packets.append(deepcopy(packet))
        result = worker.invoke({"messages": [HumanMessage(
            content=json.dumps(packet))]}, config=runtime.config)
        value = worker_result(result, result_tool)
        return Command(
            update={
                field: value, "messages": [
                    ToolMessage(
                        content=json.dumps(value), artifact=value,
                        name="schedule_event" if field == "calendar_result"
                        else "manage_email", tool_call_id=runtime.tool_call_id)
                ]
            })

    @tool
    def schedule_event(
        runtime: ToolRuntime[TrustedContext, ParentState]) -> Command:
        """Find and create a design-team event using trusted calendar
        context."""
        ctx = runtime.context
        request = next(m.content for m in reversed(runtime.state["messages"])
                       if isinstance(m, HumanMessage))
        packet = dict(request=request, attendees=list(ctx.attendees),
                      date=ctx.date, timezone=ctx.timezone,
                      reference_clock=ctx.reference_clock)
        return run_worker(calendar, packet, "create_calendar_event",
                          "calendar_result", runtime)

    @tool
    def manage_email(
        runtime: ToolRuntime[TrustedContext, ParentState]) -> Command:
        """Send a local reminder only after a calendar event exists in
        state."""
        event = runtime.state.get("calendar_result", {})
        if event.get("status") != "scheduled":
            raise ValueError("A scheduled calendar result is required first.")
        packet = dict(request="Send one design-team reminder", event=event)
        return run_worker(email, packet, "send_email", "email_result", runtime)

    supervisor = create_agent(
        factory("supervisor"), [schedule_event, manage_email],
        state_schema=ParentState, context_schema=TrustedContext,
        checkpointer=InMemorySaver(), system_prompt=
        "Schedule first. Use its result for email. Stop on rejection.",
        name="supervisor")
    return supervisor, calendar, email, workspace


## Program 5. Stream the execution and inspect real interrupt IDs

Print tool choices and tool results from actual framework updates. The collector uses subgraphs=True to capture nested worker interrupts and deduplicates their IDs. Tool artifacts remain structured even when middleware adds a human-edit notice to the message content.

Prerequisites: Run programs 1–4.

In [5]:
REQUEST = ("Find a free design-team hour after 14:00 on 2026-10-27; "
           "create the event and email a reminder.")
PRIVATE_NOTE = "PAYROLL_PRIVATE: unrelated salary discussion"

def initial_input(request=REQUEST):
    return {"messages": [HumanMessage(content=PRIVATE_NOTE),
                         HumanMessage(content=request)]}

def run(graph, value, config, verbose=True):
    pending = {}
    for namespace, update in graph.stream(value, config, context=CONTEXT,
            stream_mode="updates", subgraphs=True):
        for node, data in update.items():
            if node == "__interrupt__":
                for pause in data:
                    pending[pause.id] = pause
                continue
            if verbose and isinstance(data, dict):
                for message in data.get("messages", []):
                    if isinstance(message, AIMessage):
                        for call in message.tool_calls:
                            print("CALL", call["name"])
                    elif isinstance(message, ToolMessage):
                        print("RESULT", message.name, message.status)
    if verbose:
        for identifier, pause in pending.items():
            print("INTERRUPT", identifier)
            for action in pause.value["action_requests"]:
                print(action["name"], json.dumps(action["args"]))
    return list(pending.values())

def resume(pause, decision):
    return Command(resume={pause.id: {"decisions": [decision]}})


## Program 6. Check each layer without an approval gate

Start with computed availability, then test each worker and the whole supervisor. These fixture assertions test orchestration rather than model quality. Gating is disabled only for these local regression checks.

Prerequisites: Run programs 1–5.

In [6]:
graph, calendar, email, layer_workspace = build_system(gated=False)
packet = dict(request="Schedule review", attendees=list(CONTEXT.attendees),
              date=CONTEXT.date)
calendar_output = calendar.invoke({"messages": [HumanMessage(
    content=json.dumps(packet))]})
event = worker_result(calendar_output, "create_calendar_event")
assert event["start"] == "2026-10-27T15:00:00+02:00"
email_output = email.invoke({"messages": [HumanMessage(content=json.dumps(
    {"request": "Remind team", "event": event}))]})
assert worker_result(email_output, "send_email")["event_id"] == event["id"]
graph, _, _, combined = build_system(gated=False)
cfg = {"configurable": {"thread_id": "combined-check"}, "recursion_limit": 20}
assert run(graph, initial_input(), cfg, verbose=False) == []
assert len(combined.events) == len(combined.outbox) == 1
assert PRIVATE_NOTE not in json.dumps(combined.packets)
assert "date" in combined.packets[0] and "event" in combined.packets[1]
assert "date" not in combined.packets[1]  # No calendar input transcript.
print("Availability, calendar worker, email worker and supervisor: passed")


Availability, calendar worker, email worker and supervisor: passed


## Program 7. Pause before creating the event

The default build places approval middleware on the two worker effect tools and a saver on the supervisor. Inspect the exact proposal before continuing. The first execution reads availability but creates no records.

Prerequisites: Run programs 1–6. Every example uses a new thread and local workspace.

In [7]:
graph, _, _, workspace = build_system()
config = {"configurable": {"thread_id": "review-demo"}, "recursion_limit": 20}
pending = run(graph, initial_input(), config)
assert len(pending) == 1
assert pending[0].value["action_requests"][0][
    "name"] == "create_calendar_event"
assert workspace.events == [] and workspace.outbox == []
print("Before calendar review: 0 events, 0 local emails")


CALL schedule_event
CALL get_available_time_slots
RESULT get_available_time_slots success
CALL create_calendar_event
INTERRUPT ba48c145db5c4ef94144e67e9b38a65e
create_calendar_event {"title": "Design review", "start": "2026-10-27T15:00:00+02:00", "end": "2026-10-27T16:00:00+02:00", "attendees": ["maya@example.test", "leo@example.test"]}
Before calendar review: 0 events, 0 local emails


## Program 8. Edit the calendar action, then inspect the dependent email

This cell stands for an explicit reviewer decision after inspecting the previous output. Move the event to the next available hour. Resume using the actual interrupt ID. Email must use the event that was actually created.

Prerequisites: Run program 7 and inspect its proposed action.

In [8]:
edited_calendar = deepcopy(pending[0].value["action_requests"][0])
edited_calendar["args"].update(start="2026-10-27T16:00:00+02:00",
                               end="2026-10-27T17:00:00+02:00")
pending = run(graph, resume(pending[0], {
    "type": "edit", "edited_action": edited_calendar}), config)
assert len(workspace.events) == 1 and workspace.outbox == []
email_action = pending[0].value["action_requests"][0]
assert email_action["name"] == "send_email"
assert workspace.events[0]["start"] in email_action["args"]["body"]
assert workspace.events[0]["end"] in email_action["args"]["body"]
print("After calendar edit: 1 event, 0 local emails")


CALL create_calendar_event
RESULT create_calendar_event success
RESULT schedule_event success
CALL manage_email
CALL send_email
INTERRUPT d65772f91ba52c9bc1369b0a5a9104f0
send_email {"to": ["maya@example.test", "leo@example.test"], "subject": "Design reminder", "body": "Design review: 2026-10-27T16:00:00+02:00 to 2026-10-27T17:00:00+02:00", "event_id": "event-1"}
After calendar edit: 1 event, 0 local emails


## Program 9. Edit the email subject and verify exact effects

Make a second visible reviewer decision for the new email interrupt. Verify the subject and times stored by the backend, then inspect custom parent state. There is no automatic approval loop.

Prerequisites: Run program 8 and inspect the email recipients, subject, body and event ID.

In [9]:
edited_email = deepcopy(email_action)
edited_email["args"]["subject"] = "REVIEWED: Design reminder"
pending = run(graph, resume(pending[0], {
    "type": "edit", "edited_action": edited_email}), config)
assert pending == []
assert len(workspace.events) == len(workspace.outbox) == 1
assert workspace.outbox[0]["subject"] == "REVIEWED: Design reminder"
assert workspace.events[0]["start"] == "2026-10-27T16:00:00+02:00"
state = graph.get_state(config).values
assert state["calendar_result"] == workspace.events[0]
assert state["email_result"] == workspace.outbox[0]
assert PRIVATE_NOTE not in json.dumps(workspace.packets)
assert "event" in workspace.packets[-1] and "date" not in workspace.packets[-1]
print(json.dumps({"event": workspace.events[0],
                  "local_email": workspace.outbox[0]}, indent=2))


CALL send_email
RESULT send_email success
RESULT manage_email success
{
  "event": {
    "status": "scheduled",
    "id": "event-1",
    "title": "Design review",
    "start": "2026-10-27T16:00:00+02:00",
    "end": "2026-10-27T17:00:00+02:00",
    "attendees": [
      "maya@example.test",
      "leo@example.test"
    ],
    "timezone": "Europe/Kyiv"
  },
  "local_email": {
    "status": "sent_locally",
    "to": [
      "maya@example.test",
      "leo@example.test"
    ],
    "subject": "REVIEWED: Design reminder",
    "body": "Design review: 2026-10-27T16:00:00+02:00 to 2026-10-27T17:00:00+02:00",
    "event_id": "event-1"
  }
}


## Program 10. Approve or reject and test the absence of effects

Use separate local runs to exercise approval and rejection. Calendar rejection must stop dependent email; email rejection must preserve the approved event but create no outbox record. These are scripted test choices.

Prerequisites: Run programs 1–9.

In [10]:
rejected_graph, _, _, rejected = build_system()
reject_cfg = {"configurable": {"thread_id": "reject-calendar"}}
pauses = run(rejected_graph, initial_input(), reject_cfg, verbose=False)
pauses = run(rejected_graph, resume(pauses[0], {
    "type": "reject", "message": "Do not schedule this meeting."}),
    reject_cfg, verbose=False)
assert pauses == [] and rejected.events == [] and rejected.outbox == []

approved_graph, _, _, approved = build_system()
approve_cfg = {"configurable": {"thread_id": "approve-then-reject"}}
pauses = run(approved_graph, initial_input(), approve_cfg, verbose=False)
pauses = run(approved_graph, resume(pauses[0], {"type": "approve"}),
             approve_cfg, verbose=False)
assert len(approved.events) == 1 and approved.outbox == []
pauses = run(approved_graph, resume(pauses[0], {
    "type": "reject", "message": "Keep the event; send no reminder."}),
    approve_cfg, verbose=False)
assert pauses == [] and len(approved.events) == 1 and approved.outbox == []
assert approved_graph.get_state(approve_cfg).values["email_result"][
    "status"] == "rejected"
print("Calendar rejection: 0 events, 0 local emails")
print("Calendar approval + email rejection: 1 event, 0 local emails")


Calendar rejection: 0 events, 0 local emails
Calendar approval + email rejection: 1 event, 0 local emails


## Program 11. Optional real OpenRouter models, explicitly opt in

Replace the test double in the same builder. Install langchain-openrouter and openrouter if this optional cell is needed. Supply an external key; never put it into the notebook. This cell makes no request unless RUN_LIVE is changed to True. Inspect and review every returned interrupt manually.

Prerequisites: Run programs 1–5. Network access and a funded OpenRouter account for live mode.

In [11]:
RUN_LIVE = False
if RUN_LIVE:
    import re
    from pathlib import Path
    from langchain_openrouter import ChatOpenRouter
    from openrouter import OpenRouter

    key_text = os.environ.get("OPENROUTER_API_KEY", "")
    key_file = Path("/Users/Shared/Repos/goit_generative_and_agentic_ai/"
                    "openrouter_api.txt")
    if not key_text and key_file.is_file():
        key_text = key_file.read_text()
    match = re.search(r"sk-or-v1-[A-Za-z0-9_-]+", key_text)
    if not match:
        raise RuntimeError("Set OPENROUTER_API_KEY outside this notebook.")
    key = match.group(0)
    sdk = OpenRouter(api_key=key, timeout_ms=45_000, retry_config=None)

    def real_model(role):
        return ChatOpenRouter(client=sdk, api_key=key,
            model="google/gemini-2.5-flash", temperature=0, max_tokens=800,
            timeout=45_000, max_retries=0,
            openrouter_provider={"require_parameters": True})

    live_graph, _, _, live_workspace = build_system(real_model)
    live_cfg = {"configurable": {"thread_id": "live-review"},
                "recursion_limit": 20}
    try:
        live_pending = run(live_graph, initial_input(), live_cfg)
    except Exception as exc:
        raise RuntimeError(
            "Live call failed. Check model/tool support, "
            "credentials and connectivity; then retry.") from exc
else:
    print("Live mode disabled; no API calls or credentials are required.")


Live mode disabled; no API calls or credentials are required.


## What the checks establish

Availability is computed from the fixture. Agent tools execute in the expected
order, nested approval gates stop local effects, reviewer edits reach the
backend, and dependent email uses the resulting event. Rejection leaves the
expected records absent. Parent state contains structured evidence while worker
packets omit the unrelated salary message.

These assertions do not measure a real model's planning, tool selection,
robustness to malicious text or cost. Live model evaluation needs separate test
cases and repeated observations. A real application also needs persistent
storage, authenticated reviewers and service-specific authorization.
